In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from statsmodels.imputation.mice import MICEData
import re

'''
КОММЕНТАРИЙ:
1) для того чтобы выбрать методы заполнения надо присвоить значения переменным p_1 и p_2 в функции process_folder
2) для того чтобы поменять количество соседей в методе KNN надо поменять значение параметра k в функции knn
3) в переменную input_folder надо указать путь до датасета
'''

# функция выдает True если признак непрерывный и False иначе
def nepr_priznac(col, n, threshold = 0.15):
    # col - столбец
    # n - количество строк
    count = col.nunique() # количество различных значений в столбце
    n = n - col.isna().sum() # не учитываем количество np.nan
    
    if count / n > threshold:
        return True
    return False

def nulevoe(col):
    mask = col.isna() # возвращает Series из True и False
    col[mask] = 0 
    return col

def med(col):
    mask = col.isna() # возвращает Series из True и False
    col[mask] = col.median()
    return col

def men(col):
    mask = col.isna() # возвращает Series из True и False
    col[mask] = col.mean()
    return col

def mod(col):
    mask = col.isna() # возвращает Series из True и False
    col[mask] = col.mode()[0]
    return col

def knn(data, k = 5):
    # 1. Отделяем целевую переменную 
    target = data['Target'] if 'Target' in data.columns else None
    features = data.drop(columns=['Target'], errors='ignore')

    # 2. Нормализация
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)

    # 3. Преобразуем обратно в DataFrame (для удобства)
    scaled_df = pd.DataFrame(scaled_features, columns=features.columns)

    # 4. Применяем KNNImputer
    imputer = KNNImputer(
        n_neighbors=k,       # Количество соседей (можно настроить)
        weights='uniform',   # берем среднее значение k признаков
        metric='nan_euclidean'  # Метрика для расчета расстояний с учетом пропусков
    )
    imputed_data = imputer.fit_transform(scaled_df)

    # 5. Возвращаем данные к исходному масштабу
    imputed_features = scaler.inverse_transform(imputed_data)
    imputed_df = pd.DataFrame(imputed_features, columns=features.columns)

    # 6. Возвращаем целевую переменную (если была)
    if target is not None:
        imputed_df['Target'] = target.values
    return imputed_df


def missforest(X_miss, max_iter=100, tol=0.038, patience=3, verbose=True):
    '''
    Улучшенный MissForest:
    - Автоматическая остановка при стабилизации предсказаний.
    - Поддержка числовых и категориальных признаков.
    
    Параметры:
    - X_miss: DataFrame с пропусками.
    - max_iter: максимум итераций (на случай, если tol не достигнут).
    - tol: порог для остановки (среднее относительное изменение).
    - patience: число итераций без улучшения перед остановкой.
    - verbose: вывод лога обучения.
    '''
    # Начальная импутация средним/модой
    n = len(X_miss)
    X_imp = X_miss.copy()
    for col in X_miss.columns:
        if nepr_priznac(X_miss[col], n):  # Числовые
            X_imp[col] = men(X_miss[col])
            #X_imp[col] = X_miss[col].fillna(X_miss[col].mean())
        else:  # Категориальные
            X_imp[col] = X_miss[col].fillna(X_miss[col].mode()[0])
    
    # Итеративный процесс
    no_improvement = 0
    for iter_num in range(max_iter):
        X_imp_prev = X_imp.copy()
        total_change = 0
        n_changed = 0
        
        for col in X_miss.columns:
            if not X_miss[col].isna().any():
                continue
                
            # Разделение на наблюдаемые и пропущенные
            obs_mask = ~X_miss[col].isna()
            obs = X_imp[obs_mask]
            mis = X_imp[~obs_mask]
            
            # Выбор модели
            if (not pd.api.types.is_numeric_dtype(X_miss[col])) or \
               (X_miss[col].nunique() <= 10):  # Числовые
                model = RandomForestClassifier(n_estimators=5, min_samples_leaf=1, random_state=0)
                
            else:  # Категориальные
                model = RandomForestRegressor(n_estimators=5, min_samples_leaf=1, random_state=0)
            
            # Обучение и предсказание
            model.fit(obs.drop(col, axis=1), obs[col])
            preds = model.predict(mis.drop(col, axis=1))
            
            # Относительное изменение (для числовых) или % изменений (для категориальных)
            old_values = X_imp.loc[~obs_mask, col].copy()
            X_imp.loc[~obs_mask, col] = preds
            
            if nepr_priznac(X_miss[col], n):
                change = np.abs((preds - old_values) / (np.abs(old_values) + 1e-6)).mean()
            else:
                change = (preds != old_values).mean()
            
            total_change += change
            n_changed += 1
        
        avg_change = total_change / n_changed if n_changed > 0 else 0
        
        if verbose:
            print(f'Итерация {iter_num + 1}: avg_change = {avg_change:.6f}')
        
        # Критерий остановки
        if avg_change < tol:
            no_improvement += 1
            if no_improvement >= patience:
                if verbose:
                    print(f'Ранняя остановка: изменение < {tol} на {patience} итерациях')
                break
        else:
            no_improvement = 0
    
    return X_imp

def mice(data, n_runs=1, n_iterations=3):
    # Переименовываем столбцы, заменяя пробелы на подчёркивания
    data = data.rename(columns=lambda x: x.replace(' ', '_'))
    # Подготовка
    data = data.copy()
    data.columns = data.columns.str.replace('.', '_')
    data.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', col) for col in data.columns]
    # 1. Быстрая генерация импутаций (параллельно нельзя из-за GIL)
    imputed_dfs = []
    for _ in range(n_runs):
        imp = MICEData(data, perturbation_method='boot', k_pmm=5)
        for _ in range(n_iterations):
            imp.update_all()
        imputed_dfs.append(imp.data)
    
    # 2. Векторизованное вычисление среднего
    avg_imputed = pd.concat([df.stack() for df in imputed_dfs], axis=1)\
                   .mean(axis=1)\
                   .unstack()
    
    # 3. Подготовка ближайших значений для всех столбцов
    final_data = data.copy()
    
    for col in final_data.columns:
        missing_mask = final_data[col].isna()
        if not missing_mask.any():
            continue

        # Векторизованный поиск ближайших
        existing = final_data[col].dropna().values
        if len(existing) == 0:
            continue
            
        # Для всех пропусков сразу
        imputed_vals = avg_imputed.loc[missing_mask, col].values
        sorted_existing = np.sort(existing)
        idx = np.searchsorted(sorted_existing, imputed_vals)
        np.clip(idx, 0, len(sorted_existing)-1, out=idx)
        closest = sorted_existing[idx]
        
        # Коррекция граничных случаев
        prev_idx = np.maximum(idx - 1, 0)
        closest = np.where(
            np.abs(imputed_vals - sorted_existing[prev_idx]) < np.abs(imputed_vals - closest),
            sorted_existing[prev_idx],
            closest
        )
        
        final_data.loc[missing_mask, col] = closest
    
    return final_data
            

def zapolnenie_propuska(input_path, output_path, p_1 = 1, p_2 = 1):
    '''
    data - датасет с пропусками
    p_1 - какой метод заполнения для непрерывных признаков
    p_2 - какой метод заполнения для категориальных признаков

    для p_1:
        1 - нулевое
        2 - медиана
        3 - среднее
        4 - KNN
        5 - missForest
        6 - MICE
    для p_2:
        1 - мода
        2 - KNN
        3 - missForest
        4 - MICE
    '''
    try:
        data = pd.read_csv(input_path, dtype='float32')
        if 'Target' not in data.columns:
            raise ValueError("Отсутствует колонка 'Target'")
            
        n = len(data)
        features = [col for col in data.columns if col != 'Target']
        #data = data.astype(float)
        if p_1 == 1 and p_2 == 1:
            for col in features:
                if nepr_priznac(data[col], n):
                    data[col] = nulevoe(data[col])
                else:
                    data[col] = mod(data[col])

        elif p_1 == 2 and p_2 == 1:
            for col in features:
                if nepr_priznac(data[col], n):
                    data[col] = med(data[col])
                else:
                    data[col] = mod(data[col])

        elif p_1 == 3 and p_2 == 1:
            for col in features:
                if nepr_priznac(data[col], n):
                    data[col] = men(data[col])
                else:
                    data[col] = mod(data[col])

        elif p_1 == 4 and p_2 == 2:
            data = knn(data)

        elif p_1 == 1 and p_2 == 2:
            for col in features:
                if nepr_priznac(data[col], n):
                    data[col] = nulevoe(data[col])
            data = knn(data) # для категориальных признаков используем KNN

        elif p_1 == 2 and p_2 == 2:
            for col in features:
                if nepr_priznac(data[col], n):
                    data[col] = med(data[col])
            data = knn(data) # для категориальных признаков используем KNN

        elif p_1 == 3 and p_2 == 2:
            for col in features:
                if nepr_priznac(data[col], n):
                    data[col] = men(data[col])
            data = knn(data) # для категориальных признаков используем KNN

        elif p_1 == 4 and p_2 == 1:
            for col in features:
                if not nepr_priznac(data[col], n):
                    data[col] = mod(data[col])
            data = knn(data) # для категориальных признаков используем KNN
        
        elif p_1 == 5 and p_2 == 3:
            dataset = data.drop('Target', axis = 1)
            dataset = missforest(dataset)
            dataset = pd.concat([dataset, data['Target']], axis=1)
            data = dataset.copy()

        elif p_1 == 6 and p_2 == 4:
            data = mice(data)
            
        
        data.to_csv(output_path, index=False)
        return True
    except Exception as e:
        print(f"Ошибка в {os.path.basename(input_path)}: {str(e)}")
        return False


def process_folder(input_folder, output_folder):
    """Обрабатывает все CSV-файлы с прогресс-баром"""
    os.makedirs(output_folder, exist_ok=True)
    files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    
    with tqdm(total=len(files), desc="Обработка файлов") as pbar:
        processed_files = 0
        for filename in files:
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            p_1 = 6
            p_2 = 4
            if zapolnenie_propuska(input_path, output_path,  p_1, p_2):
                processed_files += 1
            pbar.update(1)
    
    print(f"\nГотово! Обработано {processed_files}/{len(files)} файлов")
    print(f"Результаты в: {os.path.abspath(output_folder)}")

# Установка seed для воспроизводимости
np.random.seed(42)

papki = ['MAR_5', 'MAR_15', 'MAR_30', 'MAR_50', 'MCAR_5', 'MCAR_15', 'MCAR_30', 'MCAR_50', 'MNAR_5', 'MNAR_15', 'MNAR_30', 'MNAR_50']

# Пути к данным
input_folder = r'D:\Data\MAR_15\ERROR'
output_folder = os.path.join(input_folder, 'MAR_15_MICE')

# Запуск обработки
process_folder(input_folder, output_folder)

Обработка файлов: 100%|██████████████████████████████████████████████████████████████| 6/6 [1:45:31<00:00, 1055.33s/it]


Готово! Обработано 6/6 файлов
Результаты в: D:\Data\MAR_15\ERROR\MAR_15_MICE


In [5]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

def missforest(X_miss,  max_iter=1):
    # Начальная импутация средним/модой
    X_imp = X_miss.fillna(X_miss.mean())  # Для числовых
    # Итеративный процесс
    for _ in range(max_iter):
        for col in X_miss.columns:
            # Пропускаем столбцы без NA
            if X_miss[col].isna().sum() == 0:
                continue
            # Разделение на наблюдаемые и пропущенные
            obs = X_imp.loc[~X_miss[col].isna()]
            mis = X_imp.loc[X_miss[col].isna()]
            # Обучение модели
            if pd.api.types.is_numeric_dtype(X_miss[col]):
                model = RandomForestRegressor(n_estimators=50)
            else:
                model = RandomForestClassifier(n_estimators=50)
            model.fit(obs.drop(col, axis=1), obs[col])
            # Предсказание пропусков
            X_imp.loc[X_miss[col].isna(), col] = model.predict(mis.drop(col, axis=1))
    return X_imp

dataset = pd.read_csv('abalone_binclass_binaryClass.csv')
data = dataset.drop('Target', axis = 1)     
data = data.astype(float)
data = custom_missforest(data)
data.to_csv('proverka.csv', index=False)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

def pff(real_col, preds_col):
    # Преобразуем в numpy массивы для ускорения
    arr1 = real_col.values
    arr2 = preds_col.values

    # Для каждого элемента в series2 находим ближайший в series1
    nearest_indices = np.abs(arr1 - arr2[:, None]).argmin(axis=1)
    nearest_values = real_col.iloc[nearest_indices].reset_index(drop=True)
    return nearest_values

def MICE(X_miss, method1, method2, method3, max_iter=100, tol=0.038, patience=3, verbose=True):
    '''
    Параметры:
    - X_miss: DataFrame с пропусками.
    - methodi: заполненные датасеты.
    - max_iter: максимум итераций (на случай, если tol не достигнут).
    - tol: порог для остановки (среднее относительное изменение).
    - patience: число итераций без улучшения перед остановкой.
    - verbose: вывод лога обучения.
    '''
    # Начальная импутация средним/модой
    n = len(X_miss)
    
    # Итеративный процесс
    no_improvement = 0
    for iter_num in range(max_iter):
        X_imp_prev = X_imp.copy()
        total_change = 0
        n_changed = 0
        
        for col in X_miss.columns:
            if not X_miss[col].isna().any():
                continue
                
            # Разделение на наблюдаемые и пропущенные
            obs_mask = ~X_miss[col].isna()
            obs1 = method1[obs_mask]
            mis1 = method1[~obs_mask]
            obs2 = method2[obs_mask]
            mis2 = method2[~obs_mask]
            obs3 = method3[obs_mask]
            mis3 = method3[~obs_mask]
            
            # Выбор модели
            if nepr_priznac(X_miss[col], n):  # непрерывные признаки
                model = LinearRegression()
                
            else:  # Категориальные
                model = LogisticRegression()
            
            # Обучение и предсказание
            model.fit(obs1.drop(col, axis=1), obs1[col])
            preds1 = model.predict(mis1.drop(col, axis=1))
            preds1 = pff(obs1[col], preds1)
            model.fit(obs2.drop(col, axis=1), obs2[col])
            preds2 = model.predict(mis2.drop(col, axis=1))
            preds2 = pff(obs2[col], preds2)
            model.fit(obs3.drop(col, axis=1), obs3[col])
            preds3 = model.predict(mis3.drop(col, axis=1))
            preds3 = pff(obs3[col], preds3)
            
            # Относительное изменение (для числовых) или % изменений (для категориальных)
            old_values1 = method1.loc[~obs_mask, col].copy()
            method1.loc[~obs_mask, col] = preds1
            old_values2 = method2.loc[~obs_mask, col].copy()
            method2.loc[~obs_mask, col] = preds2
            old_values3 = method3.loc[~obs_mask, col].copy()
            method3.loc[~obs_mask, col] = preds3


            # ДОДЕЛАТЬ!!!!!!!!!!!!!
            if nepr_priznac(X_miss[col], n):
                change = np.abs((preds - old_values) / (np.abs(old_values) + 1e-6)).mean()
            else:
                change = (preds != old_values).mean()
            
            total_change += change
            n_changed += 1
        
        avg_change = total_change / n_changed if n_changed > 0 else 0
        
        if verbose:
            print(f'Итерация {iter_num + 1}: avg_change = {avg_change:.6f}')
        
        # Критерий остановки
        if avg_change < tol:
            no_improvement += 1
            if no_improvement >= patience:
                if verbose:
                    print(f'Ранняя остановка: изменение < {tol} на {patience} итерациях')
                break
        else:
            no_improvement = 0

    # ДОДЕЛАТЬ НУЖНО УСРЕДНИТЬ ЗНАЧЕНИЯ ПО МЕТОДАМ 1,2,3
    
    return X_imp

In [1]:
import numpy as np
import pandas as pd
from statsmodels.imputation.mice import MICEData

class MICEDataWithTol(MICEData):
    """Расширенный MICEData с остановкой по достижению tol."""
    
    def __init__(self, data, tol=1e-3, **kwargs):
        """
        Parameters:
        -----------
        data : pd.DataFrame
            Данные с пропусками (NaN)
        tol : float
            Порог остановки
        **kwargs : 
            Доп. параметры для MICEData
        """
        # Сохраняем оригинальные данные для проверки
        self.original_data = data.copy()
        self.tol = tol
        super().__init__(data, **kwargs)
    
    def update_all(self, n_iter=10):
        """Модифицированный метод с контролем изменений."""
        # Проверяем реальные пропуски в оригинальных данных
        if not self.original_data.isnull().any().any():
            print("Внимание: В исходных данных нет пропусков!")
            return
            
        for i in range(n_iter):
            old_data = self.data.copy()
            super().update_all(n_iter=1)
            
            # Сравниваем только те ячейки, где были пропуски в оригинальных данных
            original_nans = self.original_data.isnull()
            mask = original_nans
            
            delta = np.nanmean(np.abs(self.data[mask] - old_data[mask]))
            print(f'Итерация {i+1}: Среднее изменение = {delta:.6f}')
            
            if delta < self.tol:
                print(f'>>> Остановка: изменение < {self.tol}')
                break

# -----------------------------------------------------------
# Пример использования
# -----------------------------------------------------------

# 1. Создаём данные с пропусками (обязательно используем np.nan)
data = pd.DataFrame({
    'age': [25, 30, np.nan, 40, np.nan, 28],
    'income': [5000, np.nan, 7000, np.nan, 9000, 5500],
    'employed': [1, 0, np.nan, 1, 0, 1]
}, dtype=float)  # Явно указываем float для хранения NaN

# 2. Проверяем пропуски ДО импутации
print("Пропуски в исходных данных:")
print(data.isnull().sum())

# 3. Инициализируем импутер
mice_data = MICEDataWithTol(
    data.copy(),  # Важно: передаём копию
    tol=0.01,
    perturbation_method='gaussian',
    k_pmm=5
)

# 4. Проверяем, что импутер видит пропуски
print("\nПропуски, видимые импутером:")
print(mice_data.data.isnull().sum())

# 5. Запускаем импутацию
print("\nПроцесс импутации:")
mice_data.update_all(n_iter=20)

# 6. Результаты
print("\nИмпутированные данные:")
print(mice_data.next_sample())

# 7. Проверяем пропуски после импутации
print("\nПропуски после импутации:")
print(mice_data.data.isnull().sum())

Пропуски в исходных данных:
age         2
income      2
employed    1
dtype: int64

Пропуски, видимые импутером:
age         0
income      0
employed    0
dtype: int64

Процесс импутации:
Итерация 1: Среднее изменение = 304.200000
Итерация 2: Среднее изменение = 403.000000
Итерация 3: Среднее изменение = 803.200000
Итерация 4: Среднее изменение = 802.600000
Итерация 5: Среднее изменение = 403.000000
Итерация 6: Среднее изменение = 103.200000
Итерация 7: Среднее изменение = 803.000000
Итерация 8: Среднее изменение = 403.000000
Итерация 9: Среднее изменение = 1604.600000
Итерация 10: Среднее изменение = 503.400000
Итерация 11: Среднее изменение = 1001.000000
Итерация 12: Среднее изменение = 702.200000
Итерация 13: Среднее изменение = 400.600000
Итерация 14: Среднее изменение = 300.000000
Итерация 15: Среднее изменение = 3.000000
Итерация 16: Среднее изменение = 2.400000
Итерация 17: Среднее изменение = 202.400000
Итерация 18: Среднее изменение = 2.400000
Итерация 19: Среднее изменение = 

In [1]:
import numpy as np
import pandas as pd
from statsmodels.imputation.mice import MICEData

class MICEDataWithTol(MICEData):
    """Расширенный MICEData с остановкой по достижению tol."""
    
    def __init__(self, data, tol=1e-3, **kwargs):
        """
        Parameters:
        -----------
        data : pd.DataFrame
            Данные с пропусками (NaN)
        tol : float
            Порог остановки
        **kwargs : 
            Доп. параметры для MICEData
        """
        # Сохраняем оригинальные данные для проверки
        self.original_data = data.copy()
        self.tol = tol
        super().__init__(data, **kwargs)
    
    def update_all(self, n_iter=10):
        """Модифицированный метод с контролем изменений."""
        # Проверяем реальные пропуски в оригинальных данных
        if not self.original_data.isnull().any().any():
            print("Внимание: В исходных данных нет пропусков!")
            return
            
        for i in range(n_iter):
            old_data = self.data.copy()
            super().update_all(n_iter=1)
            
            # Сравниваем только те ячейки, где были пропуски в оригинальных данных
            original_nans = self.original_data.isnull()
            mask = original_nans
            
            delta = np.nanmean(np.abs(self.data[mask] - old_data[mask]))
            print(f'Итерация {i+1}: Среднее изменение = {delta:.6f}')
            
            if delta < self.tol:
                print(f'>>> Остановка: изменение < {self.tol}')
                break

data = pd.read_csv('2dplanes_binclass_binaryClass.csv')
# Переименовываем столбцы
data.columns = data.columns.str.replace('.', '_')
# 2. Проверяем пропуски ДО импутации
print("Пропуски в исходных данных:")
print(data.isnull().sum())

# 3. Инициализируем импутер
mice_data = MICEDataWithTol(
    data.copy(),  # Важно: передаём копию
    tol=0.01,
    perturbation_method='gaussian',
    k_pmm=5
)

# 4. Проверяем, что импутер видит пропуски
print("\nПропуски, видимые импутером:")
print(mice_data.data.isnull().sum())

# 5. Запускаем импутацию
print("\nПроцесс импутации:")
mice_data.update_all(n_iter=100)

# 6. Результаты
print("\nИмпутированные данные:")
print(mice_data.next_sample())

# 7. Проверяем пропуски после импутации
print("\nПропуски после импутации:")
print(mice_data.data.isnull().sum())

Пропуски в исходных данных:
x1        12230
x2        12230
x3        12230
x4        12230
x5        12230
x6        12230
x7        12230
x8        12230
x9        12230
x10       12230
Target        0
dtype: int64

Пропуски, видимые импутером:
x1        0
x2        0
x3        0
x4        0
x5        0
x6        0
x7        0
x8        0
x9        0
x10       0
Target    0
dtype: int64

Процесс импутации:
Итерация 1: Среднее изменение = 0.645446
Итерация 2: Среднее изменение = 0.679011
Итерация 3: Среднее изменение = 0.669951
Итерация 4: Среднее изменение = 0.666460
Итерация 5: Среднее изменение = 0.650458
Итерация 6: Среднее изменение = 0.649706
Итерация 7: Среднее изменение = 0.641235
Итерация 8: Среднее изменение = 0.638479
Итерация 9: Среднее изменение = 0.637563
Итерация 10: Среднее изменение = 0.635814
Итерация 11: Среднее изменение = 0.637490
Итерация 12: Среднее изменение = 0.634350
Итерация 13: Среднее изменение = 0.634652
Итерация 14: Среднее изменение = 0.630826
Итерация 

In [2]:
import numpy as np
import pandas as pd
from statsmodels.imputation.mice import MICEData
import re

def mice(data, n_runs=1, n_iterations=3):
    # Переименовываем столбцы, заменяя пробелы на подчёркивания
    data = data.rename(columns=lambda x: x.replace(' ', '_'))
    # Подготовка
    data = data.copy()
    data.columns = data.columns.str.replace('.', '_')
    data.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', col) for col in data.columns]
    # 1. Быстрая генерация импутаций (параллельно нельзя из-за GIL)
    imputed_dfs = []
    for _ in range(n_runs):
        imp = MICEData(data, perturbation_method='boot', k_pmm=5)
        for _ in range(n_iterations):
            imp.update_all()
        imputed_dfs.append(imp.data)
    
    # 2. Векторизованное вычисление среднего
    avg_imputed = pd.concat([df.stack() for df in imputed_dfs], axis=1)\
                   .mean(axis=1)\
                   .unstack()
    
    # 3. Подготовка ближайших значений для всех столбцов
    final_data = data.copy()
    
    for col in final_data.columns:
        missing_mask = final_data[col].isna()
        if not missing_mask.any():
            continue

        # Векторизованный поиск ближайших
        existing = final_data[col].dropna().values
        if len(existing) == 0:
            continue
            
        # Для всех пропусков сразу
        imputed_vals = avg_imputed.loc[missing_mask, col].values
        sorted_existing = np.sort(existing)
        idx = np.searchsorted(sorted_existing, imputed_vals)
        np.clip(idx, 0, len(sorted_existing)-1, out=idx)
        closest = sorted_existing[idx]
        
        # Коррекция граничных случаев
        prev_idx = np.maximum(idx - 1, 0)
        closest = np.where(
            np.abs(imputed_vals - sorted_existing[prev_idx]) < np.abs(imputed_vals - closest),
            sorted_existing[prev_idx],
            closest
        )
        
        final_data.loc[missing_mask, col] = closest
    
    return final_data


'''
def mice(data):
    # Переименовываем столбцы
    data.columns = data.columns.str.replace('.', '_')
    imp = MICEData(data, perturbation_method = 'gaussian', k_pmm = 10, history_callback = None)
    for j in range(15):
        imp.update_all()
        
    return imp.data
'''

data = pd.read_csv('yprop_4_1_regression_oz252.csv')
data = mice(data)
data.to_csv('AAAyprop_4_1_regression_oz252.csv', index=False)